# DemoFast

Faster rollout than `Demo.ipynb` by replacing the Python `for` loop with `jax.lax.scan`.

**Why it's faster:** A Python loop dispatches one JAX call per step, with Python overhead between each. `jax.lax.scan` compiles the entire rollout into a single JIT-compiled GPU operation — no Python overhead, no repeated kernel launches.

The first run triggers JIT compilation (slow). Every subsequent run uses the compiled graph (fast).

In [1]:
import sys
print(sys.executable)

import jax
print("jax:", jax.__version__)

/bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/bin/python
jax: 0.8.2


In [14]:
%env MUJOCO_GL=egl

import os
import time
import mujoco
import numpy as np

import jax
import mediapy as media
from tqdm import tqdm
import twmr  # noqa: F401
import warp as wp
from jax import numpy as jp
from mujoco_playground import registry

wp.config.quiet = True

media.set_ffmpeg("/bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.pixi/envs/default/bin/ffmpeg")

xla_flags = os.environ.get("XLA_FLAGS", "")
xla_flags += " --xla_gpu_triton_gemm_any=True"
os.environ["XLA_FLAGS"] = xla_flags

env = registry.load("TWMRLegFlat")

initial_state = env.reset(jax.random.PRNGKey(0))
# action = jp.ones(env.action_size) * -1
action = jp.array([1.0, -1.0, 1.0, -1.0, 3.0, 3.0, 3.0, 3.0])

N_STEPS = 1000  # number of simulation steps


# Build a JIT-compiled scan over the episode.
# jax.lax.scan requires a static length, so N_STEPS is captured at compile time.
@jax.jit
def run_episode(initial_state, action):
    def step_fn(state, _):
        state = env.step(state=state, action=action)
        return state, state

    final_state, rollout = jax.lax.scan(step_fn, initial_state, None, length=N_STEPS)
    return final_state, rollout


# --- First call: includes JIT compilation ---
print("First call (includes JIT compilation) ...")
t0 = time.perf_counter()
final_state, rollout = run_episode(initial_state, action)
rollout.data.qpos.block_until_ready()  # wait for GPU to finish
t1 = time.perf_counter()
print(f"  {t1 - t0:.2f}s  ({N_STEPS / (t1 - t0):.0f} steps/s)")

# --- Second call: compiled graph, no Python overhead ---
print("Second call (compiled, no Python overhead) ...")
initial_state2 = env.reset(jax.random.PRNGKey(1))
t0 = time.perf_counter()
final_state2, rollout2 = run_episode(initial_state2, action)
rollout2.data.qpos.block_until_ready()
t1 = time.perf_counter()
print(f"  {t1 - t0:.2f}s  ({N_STEPS / (t1 - t0):.0f} steps/s)")

print(f"\nFinal position: {final_state.data.qpos}")

env: MUJOCO_GL=egl
First call (includes JIT compilation) ...
  3.77s  (265 steps/s)
Second call (compiled, no Python overhead) ...
  1.97s  (508 steps/s)

Final position: [-7.8282449e-03  3.7544772e-02  3.4904063e-02 -9.8878086e-01
 -5.6471559e-05 -2.1388759e-04  1.4937346e-01  2.7996773e+01
  3.4277415e+00  3.4274604e+00  3.4274638e+00 -2.8174852e+01
  3.4277425e+00  3.4274611e+00  3.4274619e+00  2.7435003e+01
  3.4277415e+00  3.4274616e+00  3.4274640e+00 -2.8283506e+01
  3.4277422e+00  3.4274611e+00  3.4274619e+00]


In [15]:
# ── Adjustable camera parameter ──────────────────────────────────────────────
camera_distance = 1.5  # meters; increase to zoom out, decrease to zoom in
# ─────────────────────────────────────────────────────────────────────────────


def render_rollout(env, rollout, camera_distance=1.5, height=480, width=640):
    """Render a scan-produced rollout (stacked State) with a follow-camera.

    `rollout` is the second return value of jax.lax.scan: each field has an
    extra leading time dimension, e.g. rollout.data.qpos has shape
    (n_steps, qpos_dim).
    """
    mj_model = env.mj_model
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(mj_model, cam)
    cam.azimuth = 135.0    # 3/4 rear view
    cam.elevation = -20.0  # slight downward angle
    cam.distance = camera_distance

    chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
    if chassis_id < 0:
        chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "root")
    if chassis_id < 0:
        chassis_id = 1  # fallback to first non-world body

    # Pull stacked arrays to CPU once (avoids repeated device transfers)
    all_qpos = np.array(rollout.data.qpos)         # (n_steps, qpos_dim)
    all_qvel = np.array(rollout.data.qvel)         # (n_steps, qvel_dim)
    all_mocap_pos = np.array(rollout.data.mocap_pos)
    all_mocap_quat = np.array(rollout.data.mocap_quat)
    all_xfrc = np.array(rollout.data.xfrc_applied)

    n_steps = all_qpos.shape[0]
    frames = []
    d = mujoco.MjData(mj_model)
    for i in tqdm(range(n_steps), desc="Rendering frames"):
        d.qpos[:] = all_qpos[i]
        d.qvel[:] = all_qvel[i]
        d.mocap_pos[:] = all_mocap_pos[i]
        d.mocap_quat[:] = all_mocap_quat[i]
        d.xfrc_applied[:] = all_xfrc[i]
        mujoco.mj_forward(mj_model, d)

        cam.lookat[:] = d.xpos[chassis_id]
        renderer.update_scene(d, camera=cam)
        frames.append(renderer.render())

    renderer.close()
    return frames


frames = render_rollout(env, rollout, camera_distance=camera_distance)

media.show_video(frames, fps=1.0 / env.dt)
media.write_video("tlwr_demo_fast.mp4", frames, fps=1.0 / env.dt)

Rendering frames: 100%|██████████| 1000/1000 [00:00<00:00, 1084.49it/s]


This browser does not support the video tag.
